<h1> PREP WORK 

<h3>Merging and Combining SAN JOSE</h3>

In [2]:
import pandas as pd
import numpy as np
import os

folder_path = "san_jose_ca"

# List all CSV files in that folder
files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
print("Found files:", files)

# Read and combine
dfs = [pd.read_csv(os.path.join(folder_path, f)) for f in files]
merged = pd.concat(dfs, ignore_index=True)

print("Merged shape:", merged.shape)

Found files: ['animalshelterdata_fy2122.csv', 'animalshelterdata_fy1920.csv', 'animalshelterdata_fy2526.csv', 'animalshelterdata_fy2021.csv', 'animalshelterdata_fy2223.csv', 'animalshelterdata_fy2425.csv', 'animalshelterdata_fy1819.csv', 'animalshelterdata_fy2324.csv']
Merged shape: (134246, 21)


In [38]:
merged.to_csv("merged_sanjose.csv", index=False) ##MERGED SAN JOSE SAVED TO JUPYTER

<h3>MERGING THE REST OF THE DATA TOGETHER</h3>

In [39]:
austin = pd.read_csv("austin_animal.csv")
long_beach = pd.read_csv("animal-shelter-intakes-and-outcomes-longbeach.csv")
san_jose = pd.read_csv("merged_sanjose.csv")

print("Austin:", austin.shape)
print("Long Beach:", long_beach.shape)
print("San Jose:", san_jose.shape)

Austin: (173775, 12)
Long Beach: (51750, 29)
San Jose: (134246, 21)


In [40]:
austin["shelter"] = "Austin"
long_beach["shelter"] = "Long Beach"
san_jose["shelter"] = "San Jose"

In [41]:
merged_all = pd.concat([austin, long_beach, san_jose], ignore_index=True, sort=False)
print("Merged dataset shape:", merged_all.shape)
merged_all.head()

merged_all.to_csv("full_animal_data_uncleaned.csv", index=False)

Merged dataset shape: (359771, 55)


<h3>STANDARDIZING COLUMN NAMES AND DATA TYPES</h3>

In [42]:
print("Austin columns:", set(austin.columns))
print("Long Beach columns:", set(long_beach.columns))
print("San Jose columns:", set(san_jose.columns))

Austin columns: {'Outcome Subtype', 'Animal ID', 'Date of Birth', 'Animal Type', 'Color', 'DateTime', 'Sex upon Outcome', 'Breed', 'shelter', 'Age upon Outcome', 'Name', 'Outcome Type', 'MonthYear'}
Long Beach columns: {'was_outcome_alive', 'Intake Type', 'geopoint', 'Intake Date', 'outcome_is_alive', 'outcome_is_other', 'intake_duration', 'DOB', 'Kennel ID', 'Outcome Subtype', 'Animal ID', 'Animal Type', 'Secondary Color', 'Outcome Date', 'shelter', 'Animal Name', 'Reason for Intake', 'Sex', 'Intake Condition', 'Crossing', 'outcome_is_dead', 'latitude', 'is_current_month', 'outcome_is_current', 'Jurisdiction', 'longitude', 'intake_is_dead', 'Primary Color', 'Intake Subtype', 'Outcome Type'}
San Jose columns: {'OutcomeDate', 'PrimaryBreed', 'IntakeCondition', 'PrimaryColor', 'DOB', 'SecondaryColor', 'IntakeDate', 'shelter', 'AnimalType', 'Age', 'Sex', 'Crossing', 'IntakeType', 'AnimalName', 'OutcomeSubtype', 'AnimalID', 'Jurisdiction', 'OutcomeCondition', 'IntakeSubtype', 'IntakeReason

In [43]:
import pandas as pd
import numpy as np

# --- Updated helpers (no infer_datetime_format; safer string handling) ---

def parse_date_series(df, candidates):
    col = next((c for c in candidates if c and c in df.columns), None)
    if col is None:
        return pd.NaT
    return pd.to_datetime(df[col], errors="coerce")

def split_color_combo(s):
    if pd.isna(s):
        return (np.nan, np.nan)
    parts = [p.strip() for p in str(s).split("/") if p is not None and str(p).strip() != ""]
    if len(parts) >= 2:
        return (parts[0], parts[1])
    return (str(s).strip(), np.nan)

def clean_sex(s):
    if pd.isna(s):
        return np.nan
    x = str(s).strip().title()
    if x in {"Male", "M"}:
        return "Male"
    if x in {"Female", "F"}:
        return "Female"
    if "Neuter" in x:
        return "Neutered Male"
    if "Spay" in x:
        return "Spayed Female"
    if x in {"Unknown", "U", "Sex Unknown"}:
        return "Unknown"
    return x

def to_clean_str(series):
    # Robust text normalizer that won’t trigger .str errors or FutureWarnings
    s = series.astype("string")  # pandas string dtype
    s = s.str.strip()
    # Replace placeholder "nan" strings and blanks with NA without downcasting warnings
    s = s.mask(s.eq("") | s.eq("nan") | s.eq("NaN") | s.eq("<NA>"))
    return s


# --- Re-define standardize() using the safer string cleaning ---

def standardize(df, shelter_name):
    df = df.copy()

    # Map candidate columns (case-sensitive first, then case-insensitive fallback)
    def get_first(cands):
        for c in cands:
            if c in df.columns:
                return c
        lowmap = {c.lower(): c for c in df.columns}
        for c in cands:
            if c.lower() in lowmap:
                return lowmap[c.lower()]
        return None

    cols = {
        "animal_id":           get_first(["Animal ID","AnimalID"]),
        "animal_type":         get_first(["Animal Type","AnimalType","type","Type","Species"]),
        "sex":                 get_first(["Sex upon Outcome","Sex"]),
        "breed":               get_first(["Breed","PrimaryBreed"]),
        "color":               get_first(["Color"]),
        "primary_color":       get_first(["Primary Color","PrimaryColor"]),
        "secondary_color":     get_first(["Secondary Color","SecondaryColor"]),
        "name":                get_first(["Name","Animal Name","AnimalName"]),
        "dob":                 get_first(["Date of Birth","DOB"]),
        "intake_date":         get_first(["Intake Date","IntakeDate"]),
        "outcome_datetime":    get_first(["DateTime"]),
        "outcome_date":        get_first(["Outcome Date","OutcomeDate"]),
        "outcome_type":        get_first(["Outcome Type","OutcomeType"]),
        "outcome_subtype":     get_first(["Outcome Subtype","OutcomeSubtype"]),
        "outcome_condition":   get_first(["OutcomeCondition"]),
        "intake_type":         get_first(["Intake Type","IntakeType"]),
        "intake_subtype":      get_first(["Intake Subtype","IntakeSubtype"]),
        "intake_condition":    get_first(["Intake Condition","IntakeCondition"]),
        "jurisdiction":        get_first(["Jurisdiction"]),
        "crossing":            get_first(["Crossing"]),
        "shelter":             get_first(["shelter"]),
    }

    out = pd.DataFrame()

    # Text-like columns (clean safely to string dtype)
    for k in ["animal_id","animal_type","breed","name","jurisdiction","crossing","outcome_type","outcome_subtype","outcome_condition","intake_type","intake_subtype","intake_condition"]:
        col = cols[k]
        if col:
            out[k] = to_clean_str(df[col])
        else:
            out[k] = pd.Series(dtype="string")

    # Sex (normalize labels)
    if cols["sex"]:
        out["sex"] = to_clean_str(df[cols["sex"]]).map(clean_sex)
    else:
        out["sex"] = pd.Series(dtype="string")

    # Colors
    if cols["primary_color"]:
        out["primary_color"] = to_clean_str(df[cols["primary_color"]])
    else:
        out["primary_color"] = pd.Series(dtype="string")

    if cols["secondary_color"]:
        out["secondary_color"] = to_clean_str(df[cols["secondary_color"]])
    else:
        out["secondary_color"] = pd.Series(dtype="string")

    if cols["color"] and out["primary_color"].isna().all() and out["secondary_color"].isna().all():
        prim, sec = zip(*df[cols["color"]].map(split_color_combo))
        out["primary_color"] = pd.Series(prim, dtype="string")
        out["secondary_color"] = pd.Series(sec, dtype="string")

    # Dates
    out["dob"] = pd.to_datetime(df[cols["dob"]], errors="coerce") if cols["dob"] else pd.NaT
    out["intake_date"] = parse_date_series(df, [cols["intake_date"]]) if cols["intake_date"] else pd.NaT
    outcome_dt = parse_date_series(df, [cols["outcome_datetime"]]) if cols["outcome_datetime"] else pd.NaT
    outcome_d  = parse_date_series(df, [cols["outcome_date"]]) if cols["outcome_date"] else pd.NaT
    out["outcome_date"] = outcome_dt if isinstance(outcome_dt, pd.Series) else pd.Series(pd.NaT, index=df.index)
    if isinstance(outcome_d, pd.Series):
        out["outcome_date"] = out["outcome_date"].fillna(outcome_d)

    # Shelter field
    if cols["shelter"]:
        out["shelter"] = to_clean_str(df[cols["shelter"]]).fillna(shelter_name)
    else:
        out["shelter"] = pd.Series([shelter_name]*len(df), dtype="string")

    # analysis_date: prefer outcome_date, else intake_date
    out["analysis_date"] = out["outcome_date"].where(~out["outcome_date"].isna(), out["intake_date"])

    # Normalize casing for some categoricals
    out["animal_type"] = out["animal_type"].str.title()

    return out

In [44]:
std_austin = standardize(austin, "Austin")
std_lb     = standardize(long_beach, "Long Beach")
std_sj     = standardize(san_jose, "San Jose")

In [45]:
merged_std = pd.concat([std_austin, std_lb, std_sj], ignore_index=True, sort=False)

In [46]:
merged_std.to_csv("full_animal_data_cleaned", index=False)

In [4]:
full_data = pd.read_csv("full_animal_data_cleaned.csv", low_memory = False)

<h3>Standardizing Outcome Types</h3>

In [5]:
full_data["outcome_type"].unique()

array(['Transfer', 'Euthanasia', nan, 'Adoption', 'Return to Owner',
       'Died', 'Missing', 'Disposal', 'Relocate', 'Rto-Adopt', 'Stolen',
       'Lost', 'EUTHANASIA', 'RESCUE', 'ADOPTION', 'RETURN TO OWNER',
       'TRANSFER', 'DIED', 'RETURN TO WILD HABITAT', 'COMMUNITY CAT',
       'FOSTER', 'SHELTER, NEUTER, RETURN', 'FOSTER TO ADOPT',
       'TRANSPORT', 'MISSING', 'DUPLICATE', 'HOMEFIRST', 'DISPOSAL',
       'RETURN TO RESCUE', 'TRAP, NEUTER, RELEASE', 'RTO', 'REQ EUTH',
       'EUTH', 'FOUND EXP', 'SPAY', 'NEUTER', 'S/N UNABLE', 'FOUND ANIM',
       'RTF', 'LOST EXP'], dtype=object)

In [6]:
import pandas as pd
import numpy as np
import re

# --- Canonical outcome categories ---
CANON = [
    "Adoption",
    "Transfer/Rescue",
    "Return to Owner",
    "Relocate/Return-to-Field",
    "Euthanasia",
    "Died (Natural/DOA)",
    "Foster",
    "Missing/Lost/Found",
    "Other/Procedure",
    'duplicate'
]

# --- Helper functions ---
def normalize_text(x: str) -> str:
    if pd.isna(x):
        return ""
    s = str(x).strip().lower()
    s = s.replace("&", " and ")
    s = re.sub(r"[^\w\s/-]", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()

def canonicalize_outcome(x: str) -> str:
    s = normalize_text(x)

    mapping = {
        "adoption": "Adoption",
        "adopt": "Adoption",
        "rto adopt": "Return to Owner",
        "return to owner": "Return to Owner",
        "rto": "Return to Owner",
        "transfer": "Transfer/Rescue",
        "rescue": "Transfer/Rescue",
        "transport": "Transfer/Rescue",
        "return to rescue": "Transfer/Rescue",
        "relocate": "Relocate/Return-to-Field",
        "return to wild habitat": "Relocate/Return-to-Field",
        "shelter neuter return": "Relocate/Return-to-Field",
        "trap neuter release": "Relocate/Return-to-Field",
        "return to field": "Relocate/Return-to-Field",
        "euthanasia": "Euthanasia",
        "req euth": "Euthanasia",
        "euth": "Euthanasia",
        "died": "Died (Natural/DOA)",
        "disposal": "Died (Natural/DOA)",
        "foster": "Foster",
        "foster to adopt": "Foster",
        "missing": "Missing/Lost/Found",
        "lost": "Missing/Lost/Found",
        "found": "Missing/Lost/Found",
        "found anim": "Missing/Lost/Found",
        "found exp": "Missing/Lost/Found",
        "lost exp": "Missing/Lost/Found",
        "stolen": "Missing/Lost/Found",
        "homefirst": "Other/Procedure",
        "sn unable": "Other/Procedure",
        "spay": "Other/Procedure",
        "neuter": "Other/Procedure",
        "community cat": "Relocate/Return-to-Field",
    }
    if s in mapping:
        return mapping[s]

    # keyword rules
    if "rto" in s:
        return "Return to Owner"
    if any(k in s for k in ["rescue", "transfer", "transport"]):
        return "Transfer/Rescue"
    if any(k in s for k in ["return to wild", "return to field", "rtf", "tnr", "snr", "relocate"]):
        return "Relocate/Return-to-Field"
    if "euth" in s:
        return "Euthanasia"
    if any(k in s for k in ["died", "disposal"]):
        return "Died (Natural/DOA)"
    if "adopt" in s:
        return "Adoption"
    if "foster" in s:
        return "Foster"
    if any(k in s for k in ["missing", "lost", "found", "stolen"]):
        return "Missing/Lost/Found"
    if s == "":
        return "Other/Procedure"
    return "Other/Procedure"

In [7]:
full_data["outcome_canonical"] = full_data["outcome_type"].map(canonicalize_outcome)
full_data["outcome_canonical"] = pd.Categorical(full_data["outcome_canonical"], categories=CANON)

print("Before standardization:")
print(full_data["outcome_type"].value_counts(dropna=False).head(20))

print("\nAfter standardization:")
print(full_data["outcome_canonical"].value_counts(dropna=False))

Before standardization:
outcome_type
Adoption                   84598
Transfer                   48689
ADOPTION                   36554
RESCUE                     35892
TRANSFER                   33993
Return to Owner            25691
DISPOSAL                   19567
Euthanasia                 10833
RTO                        10738
EUTHANASIA                  9493
EUTH                        9121
FOSTER                      6713
RTF                         6317
RETURN TO OWNER             5362
DIED                        3548
Died                        1672
NaN                         1434
SHELTER, NEUTER, RETURN     1394
Rto-Adopt                   1241
FOUND ANIM                   912
Name: count, dtype: int64

After standardization:
outcome_canonical
Adoption                    121152
Transfer/Rescue             119278
Return to Owner              43032
Euthanasia                   29490
Died (Natural/DOA)           25664
Relocate/Return-to-Field      9209
Foster                   

<h3>Fixing Animal Types</h3>

In [8]:
ANIMAL_CANON = [
    "Dog",
    "Cat",
    "Rabbit",
    "Bird",
    "Small Mammal",
    "Reptile",
    "Livestock",
    "Other",
]

def canonicalize_animal_type(x: str) -> str:
    if pd.isna(x):
        return "Other"
    s = str(x).strip().lower()

    # --- Exact matches ---
    if s in ["dog", "canine", "puppy"]:
        return "Dog"
    if s in ["cat", "feline", "kitten"]:
        return "Cat"
    if s in ["rabbit", "bunny"]:
        return "Rabbit"
    if s in ["bird", "parrot", "cockatiel", "pigeon"]:
        return "Bird"
    if s in ["guinea pig", "hamster", "ferret", "mouse", "rat", "gerbil"]:
        return "Small Mammal"
    if s in ["snake", "lizard", "turtle", "tortoise", "iguana", "reptile"]:
        return "Reptile"
    if s in ["goat", "sheep", "pig", "cow", "chicken", "rooster", "horse", "livestock"]:
        return "Livestock"
    if s in ["wild", "amphibian", "frog", "toad"]:
        return "Other"

    # --- Keyword-based fallbacks ---
    if "dog" in s or "canine" in s:
        return "Dog"
    if "cat" in s or "feline" in s:
        return "Cat"
    if any(k in s for k in ["rabbit", "bunny"]):
        return "Rabbit"
    if any(k in s for k in ["bird", "parrot", "pigeon"]):
        return "Bird"
    if any(k in s for k in ["guinea", "hamster", "ferret", "mouse", "rat", "gerbil"]):
        return "Small Mammal"
    if any(k in s for k in ["snake", "lizard", "turtle", "reptile"]):
        return "Reptile"
    if any(k in s for k in ["goat", "sheep", "pig", "cow", "chicken", "rooster", "horse"]):
        return "Livestock"

    return "Other"

# --- Apply it to your full_data DataFrame ---
full_data["animal_type_canonical"] = full_data["animal_type"].map(canonicalize_animal_type)
full_data["animal_type_canonical"] = pd.Categorical(full_data["animal_type_canonical"], categories=ANIMAL_CANON)

# --- Verify results ---
print("Before standardization:")
print(full_data["animal_type"].value_counts(dropna=False).head(20))

print("\nAfter standardization:")
print(full_data["animal_type_canonical"].value_counts(dropna=False))


Before standardization:
animal_type
Cat           175446
Dog           143314
Other          25385
Bird           11169
Wild            2447
Rabbit           992
Reptile          569
Guinea Pig       298
Livestock        148
Amphibian          3
Name: count, dtype: int64

After standardization:
animal_type_canonical
Cat             175446
Dog             143314
Other            27835
Bird             11169
Rabbit             992
Reptile            569
Small Mammal       298
Livestock          148
Name: count, dtype: int64


<h3>Adding Age Column and Age Brackets</h3>

In [9]:
import pandas as pd
import numpy as np

def to_naive_datetime(series):
    s = pd.to_datetime(series, errors="coerce", utc=True)
    return s.dt.tz_convert(None)

# Clean and standardize date columns
full_data["dob"] = to_naive_datetime(full_data.get("dob"))
full_data["outcome_date"] = to_naive_datetime(full_data.get("outcome_date"))
full_data["intake_date"] = to_naive_datetime(full_data.get("intake_date"))

# Use outcome date first, intake date second
ref_date = full_data["outcome_date"].fillna(full_data["intake_date"])

# Compute age in years (only if DOB exists)
age_days = (ref_date - full_data["dob"]).dt.days
full_data["age_years"] = (age_days / 365.25).where(age_days.notna())

# Clean impossible or missing values
full_data.loc[full_data["age_years"] < 0, "age_years"] = np.nan
full_data.loc[full_data["dob"].isna(), "age_years"] = np.nan
full_data["age_years"] = full_data["age_years"].round(2)


In [10]:
bins = [0, 1, 3, 7, 15, 30, np.inf]
labels = ["<1", "1–<3", "3–<7", "7–<15", "15–<30", "30+"]
full_data["age_group"] = pd.cut(full_data["age_years"], bins=bins, labels=labels, right=False)

SyntaxError: unterminated string literal (detected at line 3) (2045514349.py, line 3)

In [61]:
full_data.to_csv("full_animal_data.csv", index=False)

<h3> COMPLETED AS OF 11/5</h3>